# Curating BindingDB permissive activity data and pushing to a remote intake catalog

Mirrors the ChEMBL curation notebooks in this repo (`ChEMBL_pChEMBL_permissive/curate_data_permissive.ipynb`),
but pulls from [BindingDB](https://www.bindingdb.org/)'s public REST API instead of the ChEMBL SQLite
dump, since `openadmet-toolkit` has no BindingDB curator. We use the same `Intake` catalog workflow for
dissemination: https://intake.readthedocs.io/en/latest/index.html

Targets: CAR (Q14994), FXR (Q96RI1), VDR (P11473), LXR_alpha (Q13133).



No restriction on affinity type (Ki/IC50/Kd/EC50 all included); this is the BindingDB equivalent of the ChEMBL "permissive" endpoint.


## Parameters

Everything that controls this run lives in this one cell. `PUSH_TO_S3` defaults to `False`:
rerunning this notebook end to end will regenerate local parquet files but will not touch S3
or the committed catalog YAML until someone reviews the output and flips it to `True`.

In [1]:
TARGETS = {
    "CAR": "Q14994",
    "FXR": "Q96RI1",
    "VDR": "P11473",
    "LXR_alpha": "Q13133",
}

STANDARD_TYPE = None  # BindingDB affinity_type filter applied client-side; None = no filter
ENDPOINT_LABEL = 'permissive'
SOURCE = "BindingDB"
RETRIEVAL_DATE = '2026-10-08'
# BindingDB's REST API has no discrete numbered release (unlike ChEMBL); we use the
# retrieval date as the version token for the deposition directory/catalog name.
SOURCE_VERSION = RETRIEVAL_DATE.replace("-", "")
AFFINITY_CUTOFF_NM = 10_000_000  # effectively unrestricted; BindingDB requires *some* cutoff
BUCKET = "openadmet-data-public-dev"
LOCATION = f"{SOURCE}{SOURCE_VERSION}_{ENDPOINT_LABEL}"
PUSH_TO_S3 = False  # set True only after reviewing local output; never push unreviewed data


In [2]:
from pathlib import Path

import datamol as dm
import pandas as pd
import requests
from tqdm.auto import tqdm

from openadmet.toolkit.chemoinformatics.rdkit_funcs import canonical_smiles, smiles_to_inchikey

tqdm.pandas()

location_path = Path(LOCATION)
location_path.mkdir(exist_ok=True)


/Users/dwwest/miniconda3/envs/openadmet_toolkit/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Gather BindingDB data

`getLigandsByUniprots` returns every affinity record for a UniProt target up to a cutoff, with no
server-side type filter and no pagination, so we filter to `STANDARD_TYPE` client-side after the
fetch. We canonicalize with the same `openadmet.toolkit` RDKit functions the ChEMBL notebooks use,
so `OPENADMET_CANONICAL_SMILES`/`OPENADMET_INCHIKEY` are directly comparable/joinable across sources.

In [3]:
def gather_bindingdb_data_for_target(target_name: str, uniprot_id: str, standard_type: str | None):
    print(f"working on target {target_name}")
    url = (
        "https://bindingdb.org/rest/getLigandsByUniprots"
        f"?uniprot={uniprot_id}&cutoff={AFFINITY_CUTOFF_NM}&response=application/json"
    )
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    affinities = r.json().get("getLindsByUniprotsResponse", {}).get("affinities", [])
    raw = pd.DataFrame.from_records(affinities)
    if raw.empty:
        return raw, raw

    if standard_type:
        raw = raw[raw["affinity_type"] == standard_type].reset_index(drop=True)
    if raw.empty:
        return raw, raw

    print("canonicalising raw data")
    with dm.without_rdkit_log():
        raw["OPENADMET_CANONICAL_SMILES"] = raw["smile"].progress_apply(canonical_smiles)
        raw["OPENADMET_INCHIKEY"] = raw["OPENADMET_CANONICAL_SMILES"].progress_apply(smiles_to_inchikey)

    # BindingDB affinities are reported in nM, with an optional leading censoring operator
    # (e.g. ">5000", "<10"); split that out so standard_relation isn't silently discarded.
    parsed = raw["affinity"].str.extract(r"^(?P<relation>[<>]=?)?\s*(?P<value>[\d.]+)$")
    raw["standard_relation"] = parsed["relation"].fillna("=")
    raw["standard_value"] = pd.to_numeric(parsed["value"], errors="coerce")

    raw = raw.rename(columns={
        "affinity_type": "standard_type",
        "monomerid": "bindingdb_monomer_id",
        "smile": "canonical_smiles",
        "pmid": "doc_pubmed_id",
        "doi": "doc_doi",
        "query": "bindingdb_target_name_queried",
    })

    raw["OPENADMET_TARGET"] = target_name
    raw["OPENADMET_TARGET_ID"] = uniprot_id
    raw["OPENADMET_ENDPOINT"] = standard_type if standard_type else "BindingDB_permissive"
    raw["OPENADMET_UNITS"] = "nM"
    raw["OPENADMET_SOURCE"] = SOURCE
    raw["OPENADMET_SOURCE_VERSION"] = SOURCE_VERSION

    for col in ["bindingdb_monomer_id", "canonical_smiles", "standard_type", "standard_relation",
                "doc_pubmed_id", "doc_doi", "bindingdb_target_name_queried",
                "OPENADMET_CANONICAL_SMILES", "OPENADMET_INCHIKEY", "OPENADMET_TARGET",
                "OPENADMET_TARGET_ID", "OPENADMET_ENDPOINT", "OPENADMET_UNITS",
                "OPENADMET_SOURCE", "OPENADMET_SOURCE_VERSION"]:
        raw[col] = raw[col].astype("string")

    g = raw.dropna(subset=["OPENADMET_CANONICAL_SMILES", "OPENADMET_INCHIKEY"])
    aggregated = g.groupby(["OPENADMET_CANONICAL_SMILES", "OPENADMET_INCHIKEY"]).agg(
        standard_value_mean=("standard_value", "mean"),
        standard_value_median=("standard_value", "median"),
        standard_value_std=("standard_value", "std"),
        n_records=("bindingdb_monomer_id", "count"),
    ).reset_index()
    aggregated["n_records"] = aggregated["n_records"].astype("int64")
    aggregated["OPENADMET_TARGET"] = target_name
    aggregated["OPENADMET_TARGET_ID"] = uniprot_id
    aggregated["OPENADMET_ENDPOINT"] = standard_type if standard_type else "BindingDB_permissive"
    aggregated["OPENADMET_UNITS"] = "nM"
    aggregated["OPENADMET_SOURCE"] = SOURCE
    aggregated["OPENADMET_SOURCE_VERSION"] = SOURCE_VERSION
    aggregated.sort_values("n_records", ascending=False, inplace=True)

    print("smiles duplicates", aggregated["OPENADMET_CANONICAL_SMILES"].duplicated().sum())
    print("inchikey duplicates", aggregated["OPENADMET_INCHIKEY"].duplicated().sum())
    return aggregated, raw


## Main loop

Generate the data for each target and save to local parquet. Pushing to S3 and (re)building the
Intake catalog only happens if `PUSH_TO_S3` is `True` -- see the Parameters cell.

In [4]:
uris_raw = {}
uris_agg = {}
for target, uniprot_id in TARGETS.items():
    fname_agg = f"BindingDB_{ENDPOINT_LABEL}_{target}_{uniprot_id}_aggregated.parquet"
    fname_raw = f"BindingDB_{ENDPOINT_LABEL}_{target}_{uniprot_id}_raw.parquet"
    agg_path = location_path / fname_agg
    raw_path = location_path / fname_raw

    if agg_path.exists() and raw_path.exists():
        print(f"{target} already written locally, skipping")
        continue

    agg, raw = gather_bindingdb_data_for_target(target, uniprot_id, STANDARD_TYPE)
    if raw.empty:
        print(f"  no records for {target}, skipping")
        continue

    agg.reset_index(drop=True).to_parquet(agg_path, index=False)
    raw.reset_index(drop=True).to_parquet(raw_path, index=False)
    print(f"  wrote {fname_raw} (rows={len(raw)}), {fname_agg} (rows={len(agg)})")


working on target CAR


canonicalising raw data


  0%|          | 0/280 [00:00<?, ?it/s]

  2%|▏         | 5/280 [00:00<00:19, 13.88it/s]

 11%|█▏        | 32/280 [00:00<00:02, 86.11it/s]

...

 26%|██▌       | 526/2054 [00:00<00:00, 5257.79it/s]

 51%|█████     | 1052/2054 [00:00<00:00, 5233.23it/s]

 77%|███████▋  | 1579/2054 [00:00<00:00, 5245.94it/s]

100%|██████████| 2054/2054 [00:00<00:00, 5182.55it/s]

smiles duplicates 0
inchikey duplicates 0
  wrote BindingDB_permissive_LXR_alpha_Q13133_raw.parquet (rows=2054), BindingDB_permissive_LXR_alpha_Q13133_aggregated.parquet (rows=1545)


## Push to S3 and build the Intake catalog

Guarded by `PUSH_TO_S3`. Reads AWS credentials from the environment (never shell out to
`aws configure get` in a committed notebook -- see the deposition guidelines, section 6).

In [5]:
if PUSH_TO_S3:
    import intake
    from intake.readers.datatypes import Parquet

    from openadmet.toolkit.webservices.credentials import S3Settings
    from openadmet.toolkit.webservices.s3 import S3Bucket

    assert "AWS_ACCESS_KEY_ID" in __import__("os").environ, "set AWS credentials in the environment first"
    settings = S3Settings()
    bucket = S3Bucket.from_settings(settings, BUCKET)

    cat = intake.entry.Catalog()
    for target, uniprot_id in TARGETS.items():
        fname_agg = f"BindingDB_{ENDPOINT_LABEL}_{target}_{uniprot_id}_aggregated.parquet"
        fname_raw = f"BindingDB_{ENDPOINT_LABEL}_{target}_{uniprot_id}_raw.parquet"
        agg_path, raw_path = location_path / fname_agg, location_path / fname_raw
        if not (agg_path.exists() and raw_path.exists()):
            continue
        for path, fname, subset in [(agg_path, fname_agg, "aggregated"), (raw_path, fname_raw, "raw")]:
            dest = f"{LOCATION}/{fname}"
            bucket.push_file(path, dest)
            uri = bucket.to_uri(dest)
            row_desc = "one row per compound" if subset == "aggregated" else "one row per source record"
            metadata = {
                "description": (
                    f"{SOURCE}{SOURCE_VERSION} {ENDPOINT_LABEL} activity data for {target} "
                    f"({uniprot_id}), {row_desc}."
                ),
                "units": "nM",
                "endpoint": ENDPOINT_LABEL,
                "target": target,
                "target_uniprot_id": uniprot_id,
                "source_version": SOURCE_VERSION,
            }
            data = Parquet(url=uri, metadata=metadata)
            cat[f"{target}_{subset}"] = intake.readers.PandasParquet(data)

    catname = f"CATALOG_{LOCATION}.yaml"
    cat.to_yaml_file(catname)
    bucket.push_file(catname, f"{LOCATION}/{catname}")
    print(f"pushed catalog to {LOCATION}/{catname}")
else:
    print("PUSH_TO_S3 is False -- local parquet only, nothing pushed, no catalog built")


PUSH_TO_S3 is False -- local parquet only, nothing pushed, no catalog built
